### Notebook to proceed with the following steps once the yolo model has been trained. 
Will run manually here sequences of various scripts but mainly the boat_utils.testing.py

Utility functions for training/validation pipeline.  
Includes: 

    - prepare: Prepare the images for segmentation
    - segment: Segment the images
    - run_detection: Run the YoloV5 detection
    - backwards_annotation_AF: Generate labelme style annotations from the classifications
    - compare_detections_to_ground_truth: Match up labels and detections, compare them, and save the results
    - confusion_matrix_AF: Summarize the results of the comparison
    - plot_waterholes: Plot a comparison of my labels vs the model detection of waterholes on single stitched back together images. 

Load first all the configs and required packs:

In [1]:

import os
import shutil
import yaml 
import argparse
import os.path as path
import scipy.cluster
import scipy.spatial
import json
import sys
import subprocess

import numpy as np
import pandas as pd
import scipy
import random
from PIL import Image, ImageDraw
import matplotlib.pyplot as plt
from sklearn.metrics import ConfusionMatrixDisplay

from tqdm import tqdm
import torch
import stat
import shutil
from datetime import datetime

from counting_boats.boat_utils.config import cfg
from counting_boats.boat_utils import image_cutting_support as ics
from counting_boats.boat_utils import heatmap as hm

import counting_boats.boat_utils.classifier
# cluster, process_clusters, read_classifications, pixel2latlong

# Add the project root to sys.path (adjust as needed)
sys.path.append(os.path.abspath("counting_waterholes"))


c:\Users\fossatia\AppData\Local\miniconda3\envs\Boats\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
YOLOv5  v7.0-394-g86fd1ab2 Python-3.10.16 torch-1.12.1+cu113 CUDA:0 (GeForce GTX 1080, 8192MiB)

Fusing layers... 
Model summary: 157 layers, 7015519 parameters, 0 gradients, 15.8 GFLOPs
Adding AutoShape... 


If needed, extract the tif files from the zip files obtained from Planet:

In [ ]:
import os
import yaml

import counting_boats.boat_utils.planet_utils

counting_boats.boat_utils.planet_utils.extract_zip_AF(r'D:\Waterholes_project\counting_waterholes\deployment\zips\mimal_full1_20240607_psscene_analytic_sr_udm2.zip',aoi='mimal_full1', date='20240607', cfg="config_train_Drive.yaml")

Then we can select manually which function we want to run. 
First, from the testing aoi tif file, we need to prepare the padded png image using the testing.prepare():

In [5]:
# Run preparation:
import counting_boats.boat_utils.testing


counting_boats.boat_utils.testing.prepare("D:\Waterholes_project\counting_waterholes\deployment", "config_deploy_Drive.yaml")

read the file path correctly
.\.\images
D:/Waterholes_project/counting_waterholes/deployment/raw_images
Creating png for 20240607_mimal_full1.tif
Doing gdal work...
Done with gdal work for 20240607_mimal_full1.tif
New Width:  36608 New Height:  27456
Creating png for 20240607_mimal_full2.tif
Doing gdal work...
Done with gdal work for 20240607_mimal_full2.tif
New Width:  32864 New Height:  27456
Creating png for 20240607_mimal_full3.tif
Doing gdal work...
Done with gdal work for 20240607_mimal_full3.tif
New Width:  36608 New Height:  22464
Creating png for 20240607_mimal_full4.tif
Doing gdal work...
Done with gdal work for 20240607_mimal_full4.tif
New Width:  32864 New Height:  22464


Then I need to use the created png to label it with labelme. This will allow us to compare my annotation to the detection of the trained model i.e. test the model. 

Once the manual annotation is done, we can apply the segmentation used from the testing.segment(): {running time is quite long (~15minutes for 2 test images) because of the segregation by date and image for the segmented images + labels (moving them between folders)...}. 

The padded pngs and the json files should both be stored in the pngs subfolder of the testing folder. 

In [ ]:
#run segmentation without spliting 80% of the images for validation!
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.segment(r"D:/Waterholes_project/counting_waterholes/deployment", "config_deploy_Drive.yaml")

pngs folder D:\Waterholes_project\counting_waterholes\deployment\pngs
Cropping Image: D:\Waterholes_project\counting_waterholes\deployment\pngs\20240607_mimalfull2.png
[27456 32864     3]
We will have:  81693  images maximum
0% of images without labels will be removed


Saving Segments: 100%|██████████| 81693/81693 [49:14<00:00, 27.65it/s] 


Skipped 2074 images
Empty 0 images
Segregating by day...
07_06_2024
Segregating by image...


Using those segmented labelled images, we can run the detection of waterholes using the trained model, and compare my label with the detection of the model. 

Debugging section to recognise and work well with the GPU:

Note to user: Need to update the torchvision to match the cuda (GPU) version. using the 'nvidia-smi' command, you get the cuda version (my case: 11) so I need to get a version of torch and torchaudio with to 11.xx. Need to 'pip uninstall torch torchvision', and then install the correct version, in my case: 'pip install torch==1.12.1+cu113 torchvision==0.13.1+cu113 torchaudio==0.12.1 --extra-index-url https://download.pytorch.org/whl/cu113'  
Other version to be found on this website: https://pytorch.org/get-started/previous-versions/

The testing.segment function and run_detection work to use the segmented images folders grouped per date. left as it is for now but just something to bear in mind!  

In [ ]:
#check of the GPU found or not? Just for debugging
torch.cuda.is_available()

True

Need to delete the classification folder in that testing directory in case your run it and encouter an error and run it again. The function is not able to overwrite the folder. Keep an eye out to delete it before running it. 

In [ ]:
#run detection on my testing 
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.run_detection(r"D:/Waterholes_project/counting_waterholes/deployment", "config_deploy_Drive.yaml")

Weights path: D:\yolo_runs\exp_v4\weights\best.pt
Yolo path: C:/Users/fossatia/Documents/Waterholes_project/yolov5
Language used: python
img_dir: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images
root: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images
Path components: ['testing_v4_improved', 'segmented_images']
img_dir: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images
root: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images\04_06_2024
Path components: ['segmented_images', '04_06_2024']
img_dir: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images
root: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images\04_06_2024\20240604_mimal_test
Path components: ['04_06_2024', '20240604_mimal_test']
Fixed classification dir: D:\Waterholes_project\counting_waterholes\testing_v4_improved\classifications\04_06_2024\20240604_mimal_test

Use the detection output of the model on my testing images to produce labelme style annotation using the backwards_annotation():  

In [3]:
#run the annotation of the images using the detection of the model:
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.backwards_annotation_AF(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Detection directory D:/Waterholes_project/counting_waterholes/testing_v4_improved\./classifications
20240604_mimal_test_labelme_auto.json saved to D:/Waterholes_project/counting_waterholes/testing_v4_improved\./pngs\20240604_mimal_test_labelme_auto.json
20230428_example_labelme_auto.json saved to D:/Waterholes_project/counting_waterholes/testing_v4_improved\./pngs\20230428_example_labelme_auto.json


Then finally, compare the detected WH with my labeled WH:

In [4]:
#comparison of my labels with the detected WH:
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.compare_detections_to_ground_truth(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Folder directory D:\Waterholes_project\counting_waterholes\testing_v4_improved
Detection directory D:/Waterholes_project/counting_waterholes/testing_v4_improved\./classifications
Labels directory D:/Waterholes_project/counting_waterholes/testing_v4_improved\./labels
Expected label directory: D:\Waterholes_project\counting_waterholes\testing_v4_improved\labels\04_06_2024\20240604_mimal_test
Expected label directory: D:\Waterholes_project\counting_waterholes\testing_v4_improved\labels\28_04_2023\20230428_example
raw_images folder D:/Waterholes_project/counting_waterholes/testing_v4_improved\./raw_images


Create the confusion matrix which summarises the results:

In [5]:
#create the confusion matrix
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.confusion_matrix_AF(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Possible to process a single images by comparing the detections and labels for a single image, used in a function but not usefull by hand. 

In [ ]:
# #create the confusion matrix
# import counting_boats.boat_utils.testing

# counting_boats.boat_utils.testing.process_image_AF(r"D:\Waterholes_project\counting_waterholes\testing_v3\classifications", r"D:/Waterholes_project/counting_waterholes/testing_v3/labels", "config_test_Drive.yaml")

Compare the counts one by one:

In [6]:
#comparison labelled vs detected WH:
import counting_boats.boat_utils.testing

counting_boats.boat_utils.testing.waterholes_count_compare(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Now we want to plot the waterholes on the images. If you want you can stitch the image separatelly but it is also integrated directly in the next plotting function. 

In [ ]:
# import counting_boats.boat_utils.stitch_PNGs

# counting_boats.boat_utils.stitch_PNGs.stitch_AF(r"D:\Waterholes_project\counting_waterholes\testing_v4\segmented_images\04_06_2024\20240604_mimal_test",
#                                                 r"D:\Waterholes_project\counting_waterholes\testing_v4\stitching")

Now I want to plot the waterholes, modified the function that used to plot boats. Now working on it to plot the waterholes allowing for a comparison of the detected vs my labeled and highlighting if they match. 

In [7]:
#plot WH after stiching the images:

import counting_boats.boat_utils.stitch_PNGs
import counting_boats.boat_utils.testing
# from counting_boats.boat_utils.stitch_PNGs import stitch_AF

counting_boats.boat_utils.testing.plot_waterholes(r"D:/Waterholes_project/counting_waterholes/testing_v4_improved", "config_test_Drive.yaml")

Using output directory: D:\Waterholes_project\counting_waterholes\testing_v4_improved\plots
Using segmented PNG base directory: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images
Found 2 image directories
Using box size to plot of dimension: 100
Stitching images first...
Stitching images in directory: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images\04_06_2024\20240604_mimal_test with name: 20240604_mimal_test
Already stitched, output exists at D:\Waterholes_project\counting_waterholes\testing_v4_improved\plots\20240604_mimal_test_stitched.png
Stitching images in directory: D:\Waterholes_project\counting_waterholes\testing_v4_improved\segmented_images\28_04_2023\20230428_example with name: 20230428_example
Already stitched, output exists at D:\Waterholes_project\counting_waterholes\testing_v4_improved\plots\20230428_example_stitched.png
Stitched images: {'20240604_mimal_test': 'D:\\Waterholes_project\\counting_waterholes\\testin